# Fine-tuning de YOLOv8 para Detecção de Árvores em Imagem Aérea (UAV)

Este notebook documenta o treino (fine-tuning) de um modelo YOLOv8 para detectar árvores e arbustos em ortofotos de drone, a partir de um modelo pré-treinado da Ultralytics.

**Contexto do projeto**: modelos genéricos pré-treinados (DeepForest, HighResCanopyHeight) não generalizaram bem para o cenário de vegetação esparsa em pastagem degradada capturado pela imagem de drone — eles foram treinados majoritariamente em dossel florestal denso. Este notebook treina um modelo customizado, usando dados anotados manualmente a partir da própria área de interesse (via QGIS + plugin Deepness + Roboflow para anotação).

Mais contexto sobre o problema e o fluxo completo de trabalho (QGIS → exportação de tiles → anotação → treino → inferência) está no README principal do repositório.

## 1. Instalação das dependências

In [ ]:
!pip install -q ultralytics

## 2. Dataset

O dataset foi anotado manualmente no [Roboflow](https://roboflow.com) a partir de tiles exportados do raster original via QGIS (plugin Deepness), e exportado no formato YOLOv8/v9.

Esta é a **terceira versão** do dataset (`v3-merge`), que combina anotações de sessões anteriores (v1 e v2) para aumentar a quantidade de exemplos de treino.

Faça o upload do arquivo `.zip` exportado do Roboflow para o ambiente do Colab antes de rodar a célula abaixo (ou monte o Google Drive, se preferir).

In [ ]:
# Upload manual do dataset exportado do Roboflow (formato YOLOv8/v9)
from google.colab import files
uploaded = files.upload()  # selecione o .zip exportado do Roboflow

In [ ]:
!unzip -q "deteccao-arvores-v3-merge.yolov9.zip" -d dataset
!find "dataset/deteccao-arvores-v3-merge.yolov9/train/images" -type f | wc -l
!find "dataset/deteccao-arvores-v3-merge.yolov9/valid/images" -type f | wc -l
!find "dataset/deteccao-arvores-v3-merge.yolov9/test/images" -type f | wc -l

## 3. Verificação da GPU

O treino foi feito usando uma GPU Tesla T4 (gratuita, fornecida pelo Google Colab).

In [ ]:
!nvidia-smi

## 4. Treino (fine-tuning)

Partimos do checkpoint pré-treinado `yolov8m.pt` (Ultralytics, treinado no COCO) e fazemos fine-tuning para a classe única `Tree`, usando os hiperparâmetros abaixo.

| Parâmetro | Valor | Observação |
|---|---|---|
| Modelo base | `yolov8m.pt` | Ponto intermediário entre velocidade e precisão |
| Épocas | 200 | Com early stopping (`patience=100` por padrão) |
| Tamanho de imagem | 640×640 | Mesma resolução dos tiles exportados do QGIS |
| Batch size | 16 | Ajustado à memória da T4 (~15GB) |
| Cache | `True` (RAM) | Acelera o treino, já que o dataset é pequeno o suficiente |


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

results = model.train(
    data="dataset/deteccao-arvores-v3-merge.yolov9/data.yaml",
    epochs=200,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    cache=True
)

## 5. Resultado do treino

O treino parou automaticamente na **época 182** por *early stopping* (sem melhora por 100 épocas seguidas), com o melhor checkpoint salvo na **época 82**:

| Métrica | Valor |
|---|---|
| Precision (Box) | 0.604 |
| Recall (Box) | 0.442 |
| mAP50 | 0.486 |
| mAP50-95 | 0.201 |

**Nota honesta**: o `mAP50-95` relativamente baixo e o early stopping precoce sugerem que o dataset (305 imagens de treino) está próximo do limite do que é possível extrair sem anotações adicionais. Expandir o dataset é o principal caminho para melhorar esses números em trabalhos futuros.

## 6. Exportação para ONNX

Para rodar o modelo treinado no QGIS (via plugin Deepness), é necessário exportar o checkpoint `.pt` para o formato `.onnx`.

In [ ]:
from ultralytics import YOLO

model = YOLO("runs/detect/train/weights/best.pt")
model.export(format="onnx")

## 7. Download do modelo treinado

In [ ]:
from google.colab import files
files.download("runs/detect/train/weights/best.onnx")